# Algoritmos Genéticos con Fitness Sharing — Reto Computación Evolutiva

**Objetivo del reto:**  
Adaptar un AG con Fitness Sharing a una nueva función objetivo más compleja (4 picos) y analizar críticamente el impacto del parámetro `SIGMA_SHARE` en la capacidad del algoritmo para encontrar todos los óptimos.

---

## Estructura del notebook

| Sección | Contenido |
|---|---|
| 1 | Importaciones y semilla de reproducibilidad |
| 2 | Definición de ambas funciones objetivo + visualización comparativa |
| 3 | Parámetros del AG (originales, sin modificar) |
| 4 | Funciones del AG (reutilizadas para ambas funciones) |
| 5 | Función de ejecución completa del AG |
| 6 | Experimento A — función original (referencia) |
| 7 | Experimento B — función nueva con 3 valores de SIGMA_SHARE |
| 8 | Comparación visual consolidada |
| 9 | Análisis y conclusiones |

---
## Sección 1 — Importaciones y reproducibilidad

**¿Qué hace esta celda?**  
Importa las librerías necesarias y fija una semilla aleatoria global.

**¿Por qué fijar la semilla?**  
Los algoritmos genéticos son estocásticos: cada ejecución produce resultados distintos. Fijar `RANDOM_SEED` garantiza que cualquier persona que ejecute este notebook obtenga exactamente los mismos resultados, lo cual es requisito básico de reproducibilidad científica.

**¿Para qué sirve cada librería?**  
- `numpy`: operaciones vectorizadas sobre la población (cálculo de fitness, distancias)  
- `matplotlib`: visualización de la función objetivo, población final y evolución del fitness  
- `random`: generación de números aleatorios para selección, crossover y mutación

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# Semilla global para reproducibilidad
# Cambiar este valor produce resultados distintos pero igualmente reproducibles
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Semilla fijada: {RANDOM_SEED}")
print("Librerías cargadas correctamente.")

---
## Sección 2 — Funciones objetivo y visualización comparativa

### 2.1 Función objetivo ORIGINAL (referencia del PDF)

**¿Qué hace?**  
Define la función multimodal del código de referencia: suma de 3 gaussianas con picos en x=2, x=5 y x=8.

**¿Por qué gaussianas?**  
Permiten controlar con precisión la posición (`μ`), el ancho (`σ`) y la altura de cada pico. Son diferenciables y continuas, propiedades útiles para evaluar el comportamiento del niching.

### 2.2 Función objetivo NUEVA (4 picos con par cercano)

**¿Qué la hace diferente?**  
Tiene 4 picos en lugar de 3, y deliberadamente incluye un **par de picos muy cercanos** (x=4.5 y x=5.0, separados 0.5 u.) cuya distancia es **menor que `SIGMA_SHARE` = 0.75**. Esto crea el escenario de prueba exigente del reto: el algoritmo con el sigma original tenderá a fusionarlos en un único nicho.

**Tabla de picos de la función nueva:**

| Pico | Posición x | Altura | Separación al vecino más cercano | Relación con σ=0.75 |
|------|-----------|--------|----------------------------------|---------------------|
| Izquierdo | 1.5 | 0.90 | 3.0 u. (hacia x=4.5) | >> σ → fácil |
| Global | 4.5 | 1.00 | **0.5 u. (hacia x=5.0)** | **< σ → desafío** |
| Vecino cercano | 5.0 | 0.85 | **0.5 u. (hacia x=4.5)** | **< σ → desafío** |
| Derecho | 8.0 | 0.75 | 3.0 u. (hacia x=5.0) | >> σ → fácil |

In [ ]:
# ==============================================================
# 2. Definición del problema multimodal
# ==============================================================

# --------------------------------------------------------------
# PARÁMETROS DE DOMINIO (compartidos por ambas funciones)
# --------------------------------------------------------------
LOWER_BOUND = 0    # Límite inferior del espacio de búsqueda
UPPER_BOUND = 10   # Límite superior del espacio de búsqueda

# --------------------------------------------------------------
# FUNCIÓN OBJETIVO ORIGINAL  (3 picos, código de referencia PDF)
# --------------------------------------------------------------
def objective_function(x):
    """
    Función objetivo multimodal ORIGINAL (código de referencia).

    Suma de 3 gaussianas con picos bien separados:
        x = 2  →  altura 1.0  (pico principal)
        x = 5  →  altura 0.8  (pico medio)
        x = 8  →  altura 0.9  (pico alto-medio)

    Separación mínima entre picos: ~3 unidades >> SIGMA_SHARE=0.75
    El niching no enfrenta dificultad para mantenerlos separados.
    Sirve como referencia de comportamiento esperado.
    """
    x = np.array(x)
    term1 = 1.0 * np.exp(-((x - 2)**2) / (2 * 0.3**2))   # Pico alto en x=2
    term2 = 0.8 * np.exp(-((x - 5)**2) / (2 * 0.5**2))   # Pico medio en x=5
    term3 = 0.9 * np.exp(-((x - 8)**2) / (2 * 0.4**2))   # Pico alto-medio en x=8
    return term1 + term2 + term3

PEAKS_ORIGINAL = [2, 5, 8]   # Posiciones de picos (para visualización)

# --------------------------------------------------------------
# FUNCIÓN OBJETIVO NUEVA  (4 picos, par cercano < SIGMA_SHARE)
# --------------------------------------------------------------
def objective_function_new(x):
    """
    Función objetivo multimodal NUEVA con 4 picos.

    Picos:
        x = 1.5  →  altura 0.90  (aislado, izquierda)
        x = 4.5  →  altura 1.00  (máximo global)
        x = 5.0  →  altura 0.85  (vecino cercano, Δx=0.5 < σ_share)
        x = 8.0  →  altura 0.75  (aislado, derecha)

    El par x=4.5 / x=5.0 está separado 0.5 u., que es MENOR que
    SIGMA_SHARE original (0.75). Con ese sigma, ambos picos caen
    dentro del mismo radio de compartición, por lo que el algoritmo
    los tratará como un único nicho → este es el escenario de estudio.

    Se usan gaussianas más estrechas (σ=0.25) en el par cercano
    para que sean distinguibles visualmente a pesar de su proximidad.
    """
    x = np.array(x)
    term1 = 0.90 * np.exp(-((x - 1.5)**2) / (2 * 0.30**2))  # Pico izquierdo
    term2 = 1.00 * np.exp(-((x - 4.5)**2) / (2 * 0.25**2))  # Pico global
    term3 = 0.85 * np.exp(-((x - 5.0)**2) / (2 * 0.25**2))  # Vecino cercano
    term4 = 0.75 * np.exp(-((x - 8.0)**2) / (2 * 0.35**2))  # Pico derecho
    return term1 + term2 + term3 + term4

PEAKS_NEW          = [1.5, 4.5, 5.0, 8.0]  # Posiciones de picos
CLOSE_PAIR         = (4.5, 5.0)             # Par de picos problemáticos
CLOSE_PAIR_DIST    = 0.5                    # Δx = 0.5 < SIGMA_SHARE = 0.75

# --------------------------------------------------------------
# VISUALIZACIÓN COMPARATIVA DE AMBAS FUNCIONES
# --------------------------------------------------------------
x_plot     = np.linspace(LOWER_BOUND, UPPER_BOUND, 500)
y_original = objective_function(x_plot)
y_new      = objective_function_new(x_plot)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# ── Panel izquierdo: función original ──────────────────────────
axes[0].plot(x_plot, y_original, 'b-', linewidth=2)
axes[0].set_title("Función Objetivo ORIGINAL (3 picos)", fontsize=13)
axes[0].set_xlabel("x")
axes[0].set_ylabel("f(x)")
axes[0].grid(True)
for px in PEAKS_ORIGINAL:
    axes[0].axvline(px, color='gray', linestyle='--', alpha=0.5)
    axes[0].text(px + 0.05, 0.05, f'x={px}', color='gray', fontsize=9)

# ── Panel derecho: función nueva ───────────────────────────────
axes[1].plot(x_plot, y_new, 'r-', linewidth=2)
axes[1].set_title(
    f"Función Objetivo NUEVA (4 picos | par cercano Δx={CLOSE_PAIR_DIST} < σ=0.75)",
    fontsize=13
)
axes[1].set_xlabel("x")
axes[1].set_ylabel("f(x)")
axes[1].grid(True)
for px in PEAKS_NEW:
    es_cercano = px in CLOSE_PAIR
    color  = 'crimson' if es_cercano else 'gray'
    estilo = '-.'      if es_cercano else '--'
    axes[1].axvline(px, color=color, linestyle=estilo, alpha=0.6)
    axes[1].text(px + 0.05, 0.05, f'x={px}', color=color, fontsize=9)

# Anotación del par cercano
axes[1].annotate(
    f'Par cercano\nΔx={CLOSE_PAIR_DIST} < σ=0.75',
    xy=((CLOSE_PAIR[0] + CLOSE_PAIR[1]) / 2, 0.55),
    xytext=(6.3, 0.75),
    arrowprops=dict(arrowstyle='->', color='crimson'),
    color='crimson', fontsize=9
)

plt.suptitle(
    "Comparación de funciones objetivo — Original vs. Nueva",
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.show()

---
## Sección 3 — Parámetros del Algoritmo Genético

**¿Qué hace esta celda?**  
Define todas las constantes que controlan el comportamiento del AG. Se mantienen **exactamente iguales al código de referencia del PDF** para no introducir variables adicionales que contaminen la comparación.

**¿Por qué centralizar los parámetros aquí?**  
Tener todos los parámetros en un único lugar permite modificar cualquier valor y que el cambio se propague automáticamente a todas las ejecuciones. Evita el antipatrón de "números mágicos" dispersos en el código.

**Parámetro clave para el reto — `SIGMA_SHARE`:**  
Define el radio del nicho. Dos individuos separados menos de `SIGMA_SHARE` se consideran del mismo nicho y se penalizan mutuamente. Es el parámetro que se experimenta en la Sección 7.

In [ ]:
# ==============================================================
# 3. Parámetros del Algoritmo Genético
# ==============================================================
# Estos valores son exactamente los del código de referencia (PDF).
# Solo SIGMA_SHARE se variará en la Sección 7 del experimento.
# ==============================================================

# ── Parámetros poblacionales ───────────────────────────────────
POPULATION_SIZE  = 100   # Individuos por generación
NUM_GENERATIONS  = 100   # Número de generaciones a evolucionar

# ── Parámetros de operadores genéticos ────────────────────────
MUTATION_RATE     = 0.1   # Probabilidad de que un individuo mute
MUTATION_STRENGTH = 0.5   # Desviación estándar de la mutación gaussiana
TOURNAMENT_SIZE   = 3     # Individuos que compiten en cada torneo
CROSSOVER_RATE    = 0.8   # Probabilidad de que dos padres se crucen

# ── Parámetros de Fitness Sharing (niching) ───────────────────
SIGMA_SHARE    = 0.75  # Radio del nicho ORIGINAL (valor de referencia del PDF)
                       # Distancia máxima para que dos individuos compartan nicho.
                       # Este valor es > CLOSE_PAIR_DIST (0.5), por lo que con
                       # este sigma el AG NO puede separar el par cercano.
ALPHA_SHARING  = 1.0   # Exponente de la función de compartición sh(d).
                       # Con α=1 la función es lineal dentro del radio.

# ── Resumen de parámetros ──────────────────────────────────────
print("=" * 50)
print("PARÁMETROS DEL AG")
print("=" * 50)
print(f"  Población:          {POPULATION_SIZE} individuos")
print(f"  Generaciones:       {NUM_GENERATIONS}")
print(f"  Tasa de mutación:   {MUTATION_RATE}")
print(f"  Fuerza de mutación: {MUTATION_STRENGTH}")
print(f"  Tamaño de torneo:   {TOURNAMENT_SIZE}")
print(f"  Tasa de crossover:  {CROSSOVER_RATE}")
print("-" * 50)
print(f"  SIGMA_SHARE:        {SIGMA_SHARE}  ← radio del nicho original")
print(f"  ALPHA_SHARING:      {ALPHA_SHARING}")
print(f"  Par cercano Δx:     {CLOSE_PAIR_DIST}  ← < SIGMA_SHARE")
print("=" * 50)

---
## Sección 4 — Funciones del Algoritmo Genético

**¿Qué hace esta sección?**  
Define todos los componentes del AG como funciones reutilizables. Al estar parametrizadas (reciben `sigma_share`, `objective_fn`, etc.), las **mismas funciones sirven tanto para la función original como para la nueva**, evitando duplicación de código.

**Flujo del AG:**
```
Inicializar población
    └─► Para cada generación:
            1. Calcular fitness original   → calculate_fitness()
            2. Aplicar Fitness Sharing     → apply_fitness_sharing()
            3. Selección por torneo        → tournament_selection()  (usa shared fitness)
            4. Crossover aritmético        → crossover()
            5. Mutación gaussiana          → mutate()
```

In [ ]:
# ==============================================================
# 4. Funciones del Algoritmo Genético
# ==============================================================
# Todas las funciones están parametrizadas para poder reutilizarse
# con cualquier función objetivo y cualquier valor de sigma_share.
# ==============================================================

# --------------------------------------------------------------
# 4.1 Inicialización
# ¿Por qué uniforme? Garantiza exploración equitativa del espacio.
# No queremos sesgar la búsqueda hacia ninguna región inicial.
# --------------------------------------------------------------
def initialize_population(size, lower_bound, upper_bound):
    """Crea una población inicial de `size` individuos distribuidos
    uniformemente en [lower_bound, upper_bound]."""
    return [random.uniform(lower_bound, upper_bound) for _ in range(size)]


# --------------------------------------------------------------
# 4.2 Evaluación de fitness
# Aplica la función objetivo a cada individuo de la población.
# Retorna una lista paralela: fitness[i] corresponde a population[i].
# --------------------------------------------------------------
def calculate_fitness(population, objective_fn):
    """Evalúa cada individuo con `objective_fn` y retorna la lista de fitness."""
    return [objective_fn(ind) for ind in population]


# --------------------------------------------------------------
# 4.3 Distancia fenotípica
# Para un problema 1D la distancia es simplemente |x_i - x_j|.
# Esta función se separa para facilitar extensión a espacios n-D.
# --------------------------------------------------------------
def phenotypic_distance(ind1, ind2):
    """Distancia fenotípica entre dos individuos (diferencia absoluta en 1D)."""
    return abs(ind1 - ind2)


# --------------------------------------------------------------
# 4.4 Fitness Sharing (núcleo del niching)
# ¿Cómo funciona?
#   Para cada individuo i, se calcula su "niche_count" = Σ sh(d_ij)
#   donde sh(d) = 1 - (d/σ)^α  si d < σ,  y  0 en caso contrario.
#   Luego: shared_fitness_i = fitness_i / niche_count_i
#
# ¿Por qué penalizar por niche_count?
#   Si muchos individuos se acumulan en el mismo pico, cada uno
#   tiene un niche_count alto → su fitness compartido baja.
#   Esto libera presión selectiva para que otros individuos
#   exploren picos menos poblados (diversidad multimodal).
# --------------------------------------------------------------
def apply_fitness_sharing(population, fitness_values, sigma_share, alpha_sharing):
    """
    Aplica Fitness Sharing y retorna la lista de fitness compartidos.

    Parámetros:
        population    : lista de individuos (valores x)
        fitness_values: fitness original de cada individuo
        sigma_share   : radio del nicho (umbral de distancia)
        alpha_sharing : exponente de la función sh(d) (1.0 = lineal)
    """
    shared_fitness_values = []
    n = len(population)

    for i in range(n):
        niche_count = 0
        for j in range(n):
            d = phenotypic_distance(population[i], population[j])
            if d < sigma_share:
                # Contribución del individuo j al nicho de i
                sh_d = 1 - (d / sigma_share) ** alpha_sharing
                niche_count += sh_d
            # Si d >= sigma_share, sh(d) = 0 → no contribuye

        # Evitar división por cero (niche_count siempre >= 1 porque sh(0)=1 cuando i=j)
        if niche_count > 1e-6:
            shared_fitness_values.append(fitness_values[i] / niche_count)
        else:
            shared_fitness_values.append(fitness_values[i])

    return shared_fitness_values


# --------------------------------------------------------------
# 4.5 Selección por torneo
# ¿Por qué torneo? Es robusto, simple y controlable con k.
# CLAVE: usa el fitness COMPARTIDO (no el original) para que
# la presión selectiva respete los nichos formados.
# --------------------------------------------------------------
def tournament_selection(population, shared_fitness, k):
    """
    Selecciona un individuo por torneo de tamaño k.
    Compara individuos usando el fitness COMPARTIDO.
    """
    indices = random.sample(range(len(population)), k)
    ganador = max(indices, key=lambda i: shared_fitness[i])
    return population[ganador]


# --------------------------------------------------------------
# 4.6 Crossover aritmético
# Genera dos hijos como combinación convexa de los padres.
# α aleatorio en [0,1] → cada cruce produce mezclas distintas.
# Preserva que los hijos queden en el rango de sus padres.
# --------------------------------------------------------------
def crossover(parent1, parent2):
    """Crossover aritmético: combina dos padres con peso aleatorio α."""
    alpha = random.random()
    child1 = alpha * parent1 + (1 - alpha) * parent2
    child2 = (1 - alpha) * parent1 + alpha * parent2
    return child1, child2


# --------------------------------------------------------------
# 4.7 Mutación gaussiana
# Añade ruido gaussiano centrado en 0 con σ = mutation_strength.
# ¿Por qué gaussiana? Perturbaciones pequeñas son más probables
# que grandes, favoreciendo exploración local alrededor del pico.
# Se aplica clipping para mantener al individuo dentro de los límites.
# --------------------------------------------------------------
def mutate(individual, mutation_rate, mutation_strength, lower_bound, upper_bound):
    """Muta un individuo con probabilidad mutation_rate usando ruido gaussiano."""
    if random.random() < mutation_rate:
        individual += random.gauss(0, mutation_strength)
        individual = max(lower_bound, min(upper_bound, individual))
    return individual


print("Funciones del AG definidas correctamente.")
print("  · initialize_population")
print("  · calculate_fitness")
print("  · phenotypic_distance")
print("  · apply_fitness_sharing")
print("  · tournament_selection")
print("  · crossover")
print("  · mutate")

---
## Sección 5 — Función de ejecución completa del AG

**¿Qué hace esta función?**  
Encapsula el ciclo completo del AG (inicialización → evolución → resultados) en una sola función llamable. Esto permite ejecutar el AG múltiples veces con distintas configuraciones (función objetivo, sigma_share) con una sola línea de código, lo que es esencial para el experimento comparativo de la Sección 7.

**¿Por qué encapsular en una función y no correr el ciclo directamente?**  
Si el ciclo principal estuviera suelto en una celda, para comparar 4 escenarios distintos habría que copiar y pegar el mismo código 4 veces, introduciendo riesgo de inconsistencias. Una función parametrizada garantiza que todos los escenarios corren exactamente la misma lógica.

In [ ]:
# ==============================================================
# 5. Función de ejecución del AG
# ==============================================================
# Encapsula el ciclo completo del AG para poder invocarlo con
# distintos valores de sigma_share y objective_fn sin repetir código.
# ==============================================================

def run_AG(
    objective_fn,
    sigma_share,
    population_size  = POPULATION_SIZE,
    num_generations  = NUM_GENERATIONS,
    mutation_rate    = MUTATION_RATE,
    mutation_strength= MUTATION_STRENGTH,
    tournament_size  = TOURNAMENT_SIZE,
    crossover_rate   = CROSSOVER_RATE,
    alpha_sharing    = ALPHA_SHARING,
    lower_bound      = LOWER_BOUND,
    upper_bound      = UPPER_BOUND,
    seed             = RANDOM_SEED,
    verbose          = True
):
    """
    Ejecuta el AG completo con Fitness Sharing.

    Parámetros principales:
        objective_fn  : función objetivo a optimizar
        sigma_share   : radio del nicho (el parámetro a experimentar)
        verbose       : si True, imprime progreso cada 20 generaciones

    Retorna un diccionario con:
        'population'        : población final
        'final_fitness'     : fitness original de la población final
        'history_best'      : mejor fitness por generación
        'history_avg'       : fitness promedio por generación
        'potential_optima_x': posiciones de picos encontrados (heurística)
        'potential_optima_y': altura de cada pico encontrado
    """
    # Fijar semilla para reproducibilidad de esta ejecución específica
    random.seed(seed)
    np.random.seed(seed)

    # ── Inicialización ─────────────────────────────────────────
    population = initialize_population(population_size, lower_bound, upper_bound)
    history_best = []
    history_avg  = []

    if verbose:
        print(f"Iniciando AG | σ_share={sigma_share} | "
              f"{num_generations} generaciones | "
              f"población={population_size}")

    # ── Ciclo evolutivo ────────────────────────────────────────
    for generation in range(num_generations):

        # 1. Fitness original (sin penalización)
        original_fitness = calculate_fitness(population, objective_fn)

        # 2. Fitness compartido (con penalización por nicho)
        shared_fitness = apply_fitness_sharing(
            population, original_fitness, sigma_share, alpha_sharing
        )

        # 3. Guardar estadísticas (en fitness ORIGINAL para ver progreso real)
        history_best.append(np.max(original_fitness))
        history_avg.append(np.mean(original_fitness))

        if verbose and (generation + 1) % 20 == 0:
            print(f"  Gen {generation+1:3d}/{num_generations} "
                  f"| Mejor: {history_best[-1]:.4f} "
                  f"| Promedio: {history_avg[-1]:.4f}")

        # 4. Selección, crossover y mutación → nueva población
        new_population = []
        while len(new_population) < population_size:
            # Selección basada en fitness COMPARTIDO (clave del niching)
            parent1 = tournament_selection(population, shared_fitness, tournament_size)
            parent2 = tournament_selection(population, shared_fitness, tournament_size)

            if random.random() < crossover_rate:
                child1, child2 = crossover(parent1, parent2)
            else:
                child1, child2 = parent1, parent2

            child1 = mutate(child1, mutation_rate, mutation_strength, lower_bound, upper_bound)
            child2 = mutate(child2, mutation_rate, mutation_strength, lower_bound, upper_bound)

            new_population.append(child1)
            if len(new_population) < population_size:
                new_population.append(child2)

        population = new_population

    # ── Post-procesamiento: identificar picos encontrados ──────
    # Heurística del PDF: tomar los mejores individuos asegurando
    # que estén separados al menos sigma_share entre sí.
    final_fitness   = calculate_fitness(population, objective_fn)
    sorted_indices  = np.argsort(final_fitness)[::-1]
    potential_optima_x = []
    potential_optima_y = []

    for i in sorted_indices:
        x_cand = population[i]
        y_cand = final_fitness[i]
        if y_cand < 0.1:   # Ignorar individuos de fitness muy bajo
            continue
        # ¿Está suficientemente lejos de todos los óptimos ya registrados?
        if all(phenotypic_distance(x_cand, ox) >= sigma_share
               for ox in potential_optima_x):
            potential_optima_x.append(x_cand)
            potential_optima_y.append(y_cand)
        if len(potential_optima_x) >= 6:   # Limitar a 6 picos mostrados
            break

    if verbose:
        print(f"\nÓptimos encontrados ({len(potential_optima_x)}):")
        for xv, yv in zip(potential_optima_x, potential_optima_y):
            print(f"  x = {xv:.4f}  →  f(x) = {yv:.4f}")

    return {
        'population':         population,
        'final_fitness':      final_fitness,
        'history_best':       history_best,
        'history_avg':        history_avg,
        'potential_optima_x': potential_optima_x,
        'potential_optima_y': potential_optima_y,
    }


print("Función run_AG() definida correctamente.")

---
## Sección 6 — Experimento A: Función ORIGINAL (caso de referencia)

**¿Qué hace esta sección?**  
Ejecuta el AG con la función objetivo original del PDF usando `SIGMA_SHARE = 0.75`. Esto establece la **línea base** de comportamiento esperado: el algoritmo debería identificar correctamente los 3 picos bien separados.

**¿Para qué sirve este experimento?**  
Confirmar que el AG con los parámetros originales funciona correctamente antes de introducir la nueva función. Si este experimento falla, el problema está en el código base, no en la función nueva.

In [ ]:
# ==============================================================
# 6. Experimento A — Función ORIGINAL (referencia)
# ==============================================================
# σ_share = 0.75 (valor original del PDF)
# Resultado esperado: 3 nichos estables en x≈2, x≈5, x≈8
# ==============================================================

print("EXPERIMENTO A — Función original | σ_share = 0.75")
print("=" * 55)

resultado_original = run_AG(
    objective_fn = objective_function,
    sigma_share  = 0.75,
    verbose      = True
)

# ── Visualización ──────────────────────────────────────────────
x_plot = np.linspace(LOWER_BOUND, UPPER_BOUND, 500)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9))

# Panel 1: función objetivo + población final
ax1.plot(x_plot, objective_function(x_plot), 'b-', linewidth=2, label='Función Objetivo')
ax1.scatter(
    resultado_original['population'],
    resultado_original['final_fitness'],
    color='red', s=30, alpha=0.7, label='Población Final'
)
ax1.scatter(
    resultado_original['potential_optima_x'],
    resultado_original['potential_optima_y'],
    color='green', s=150, marker='*', edgecolor='black',
    zorder=5, label='Óptimos Encontrados'
)
ax1.set_title("Exp. A — Función Original | σ_share=0.75 | Población Final", fontsize=12)
ax1.set_xlabel("x")
ax1.set_ylabel("f(x)")
ax1.legend()
ax1.grid(True)

# Panel 2: evolución del fitness
ax2.plot(resultado_original['history_best'], color='green', label='Mejor Fitness (original)')
ax2.plot(resultado_original['history_avg'],  color='orange', label='Fitness Promedio (original)')
ax2.set_title("Exp. A — Evolución del Fitness", fontsize=12)
ax2.set_xlabel("Generación")
ax2.set_ylabel("Fitness")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

---
## Sección 7 — Experimento B: Función NUEVA con 3 valores de SIGMA_SHARE

**¿Qué hace esta sección?**  
Ejecuta el AG con la nueva función objetivo (4 picos) usando tres valores distintos de `SIGMA_SHARE`, tal como lo requiere el objetivo 2 del reto.

**Diseño del experimento:**

| Escenario | SIGMA_SHARE | Relación con Δx=0.5 | Comportamiento esperado |
|-----------|-------------|----------------------|-------------------------|
| B1 | **2.0** | σ >> Δx (4× mayor) | Fusiona par cercano + posiblemente picos alejados |
| B2 | **0.75** | σ > Δx (1.5× mayor) | Fusiona el par cercano en un solo nicho |
| B3 | **0.25** | σ < Δx (0.5× menor) | Capaz de separar el par cercano |

**¿Por qué estos tres valores?**  
- B1 (σ=2.0): significativamente mayor que la distancia entre picos cercanos → máxima fusión  
- B2 (σ=0.75): el valor original, apenas mayor que Δx=0.5 → fusión del par cercano  
- B3 (σ=0.25): significativamente menor que Δx=0.5, pero no tan pequeño que cada individuo sea su propio nicho (cumple la condición del PDF)

In [ ]:
# ==============================================================
# 7. Experimento B — Función NUEVA con 3 valores de SIGMA_SHARE
# ==============================================================

# Configuración de los 3 escenarios del reto
# Cada escenario es un diccionario con los metadatos del experimento
escenarios = [
    {
        'id':          'B1',
        'sigma':       2.0,
        'descripcion': 'σ >> Δx  (fusión máxima, demasiado grande)',
        'color':       'darkorange',
    },
    {
        'id':          'B2',
        'sigma':       0.75,
        'descripcion': 'σ > Δx   (valor original, fusiona par cercano)',
        'color':       'royalblue',
    },
    {
        'id':          'B3',
        'sigma':       0.25,
        'descripcion': 'σ < Δx   (capaz de separar el par cercano)',
        'color':       'seagreen',
    },
]

resultados_nuevos = {}   # Almacena resultados de cada escenario para la Sección 8

for esc in escenarios:
    print(f"\n{'='*60}")
    print(f"EXPERIMENTO {esc['id']} — σ_share = {esc['sigma']}")
    print(f"{esc['descripcion']}")
    print('='*60)

    resultado = run_AG(
        objective_fn = objective_function_new,
        sigma_share  = esc['sigma'],
        verbose      = True
    )
    resultados_nuevos[esc['id']] = resultado

    # ── Visualización individual por escenario ─────────────────
    x_plot = np.linspace(LOWER_BOUND, UPPER_BOUND, 500)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9))

    # Panel 1: función objetivo + población final
    ax1.plot(
        x_plot, objective_function_new(x_plot),
        color='black', linewidth=2, label='Función Objetivo Nueva'
    )
    # Marcar picos teóricos con líneas de referencia
    for px in PEAKS_NEW:
        es_cercano = px in CLOSE_PAIR
        ax1.axvline(
            px,
            color='crimson' if es_cercano else 'lightgray',
            linestyle='--', alpha=0.5,
            label='Par cercano (teórico)' if es_cercano and px == CLOSE_PAIR[0] else ''
        )
    ax1.scatter(
        resultado['population'],
        resultado['final_fitness'],
        color='red', s=30, alpha=0.6, label='Población Final'
    )
    ax1.scatter(
        resultado['potential_optima_x'],
        resultado['potential_optima_y'],
        color=esc['color'], s=180, marker='*',
        edgecolor='black', zorder=5,
        label=f'Óptimos encontrados ({len(resultado["potential_optima_x"])})'
    )
    ax1.set_title(
        f"Exp. {esc['id']} — Función Nueva | σ_share={esc['sigma']} | "
        f"{esc['descripcion']}",
        fontsize=11
    )
    ax1.set_xlabel("x")
    ax1.set_ylabel("f(x)")
    ax1.legend(fontsize=9)
    ax1.grid(True)

    # Panel 2: evolución del fitness
    ax2.plot(resultado['history_best'], color=esc['color'], label='Mejor Fitness (original)')
    ax2.plot(resultado['history_avg'],  color='orange',     label='Fitness Promedio (original)')
    ax2.set_title(f"Exp. {esc['id']} — Evolución del Fitness | σ_share={esc['sigma']}", fontsize=11)
    ax2.set_xlabel("Generación")
    ax2.set_ylabel("Fitness")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

---
## Sección 8 — Comparación visual consolidada

**¿Qué hace esta sección?**  
Muestra los 4 experimentos (A + B1 + B2 + B3) en una única figura de 4 paneles, facilitando la comparación directa del efecto de `SIGMA_SHARE` sobre la distribución final de la población.

**¿Por qué esta visualización adicional?**  
Las figuras individuales de las secciones 6 y 7 muestran el detalle de cada experimento, pero para el análisis comparativo es más efectivo ver todos los resultados lado a lado en la misma escala.

In [ ]:
# ==============================================================
# 8. Comparación visual consolidada — 4 experimentos
# ==============================================================

x_plot = np.linspace(LOWER_BOUND, UPPER_BOUND, 500)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

# Configuración de los 4 paneles en orden
paneles = [
    {
        'titulo':    'Exp. A — Función ORIGINAL | σ=0.75',
        'resultado': resultado_original,
        'fn':        objective_function,
        'peaks':     PEAKS_ORIGINAL,
        'close':     [],
        'color':     'royalblue',
        'fn_color':  'blue',
    },
    {
        'titulo':    'Exp. B1 — Función NUEVA | σ=2.0 (demasiado grande)',
        'resultado': resultados_nuevos['B1'],
        'fn':        objective_function_new,
        'peaks':     PEAKS_NEW,
        'close':     list(CLOSE_PAIR),
        'color':     'darkorange',
        'fn_color':  'black',
    },
    {
        'titulo':    'Exp. B2 — Función NUEVA | σ=0.75 (valor original)',
        'resultado': resultados_nuevos['B2'],
        'fn':        objective_function_new,
        'peaks':     PEAKS_NEW,
        'close':     list(CLOSE_PAIR),
        'color':     'royalblue',
        'fn_color':  'black',
    },
    {
        'titulo':    'Exp. B3 — Función NUEVA | σ=0.25 (adecuado)',
        'resultado': resultados_nuevos['B3'],
        'fn':        objective_function_new,
        'peaks':     PEAKS_NEW,
        'close':     list(CLOSE_PAIR),
        'color':     'seagreen',
        'fn_color':  'black',
    },
]

for ax, panel in zip(axes, paneles):
    res = panel['resultado']

    # Función objetivo
    ax.plot(x_plot, panel['fn'](x_plot),
            color=panel['fn_color'], linewidth=1.5, alpha=0.7, label='f(x)')

    # Líneas de referencia de picos teóricos
    for px in panel['peaks']:
        es_cercano = px in panel['close']
        ax.axvline(px,
                   color='crimson' if es_cercano else 'lightgray',
                   linestyle='--', alpha=0.4, linewidth=1)

    # Población final
    ax.scatter(res['population'], res['final_fitness'],
               color='red', s=15, alpha=0.5, label='Pob. final')

    # Óptimos encontrados
    n_opt = len(res['potential_optima_x'])
    ax.scatter(res['potential_optima_x'], res['potential_optima_y'],
               color=panel['color'], s=150, marker='*',
               edgecolor='black', zorder=5,
               label=f'Óptimos: {n_opt}')

    ax.set_title(panel['titulo'], fontsize=10, fontweight='bold')
    ax.set_xlabel("x", fontsize=9)
    ax.set_ylabel("f(x)", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)
    ax.set_xlim(LOWER_BOUND, UPPER_BOUND)

plt.suptitle(
    "Comparación consolidada — Impacto de SIGMA_SHARE en la identificación de picos",
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

---
## Sección 9 — Análisis y conclusiones

**¿Qué hace esta sección?**  
Genera automáticamente una tabla resumen con los resultados de todos los experimentos y presenta el análisis requerido por el reto.

In [ ]:
# ==============================================================
# 9. Tabla resumen de resultados
# ==============================================================

print("=" * 70)
print("RESUMEN DE EXPERIMENTOS")
print("=" * 70)
print(f"{'Exp.':<6} {'Función':<12} {'σ_share':<10} {'Picos':<8} {'Posiciones encontradas'}")
print("-" * 70)

resumen = [
    ('A',  'Original', 0.75, resultado_original),
    ('B1', 'Nueva',    2.0,  resultados_nuevos['B1']),
    ('B2', 'Nueva',    0.75, resultados_nuevos['B2']),
    ('B3', 'Nueva',    0.25, resultados_nuevos['B3']),
]

for exp_id, fn_name, sigma, res in resumen:
    n_picos   = len(res['potential_optima_x'])
    posiciones = ', '.join(f"{x:.2f}" for x in sorted(res['potential_optima_x']))
    print(f"{exp_id:<6} {fn_name:<12} {sigma:<10} {n_picos:<8} {posiciones}")

print("=" * 70)
print(f"\nPicos TEÓRICOS función original : {PEAKS_ORIGINAL}")
print(f"Picos TEÓRICOS función nueva    : {PEAKS_NEW}")
print(f"Par cercano                     : x={CLOSE_PAIR[0]} y x={CLOSE_PAIR[1]} (Δx={CLOSE_PAIR_DIST})")

---
### Análisis del impacto de SIGMA_SHARE

#### Experimento A — Función original, σ=0.75
Con la función de referencia y el sigma original, el AG identifica los **3 picos correctamente**. La separación mínima entre picos (~3 u.) es mucho mayor que σ=0.75, por lo que el Fitness Sharing forma 3 nichos estables sin dificultad. Este resultado valida que el código base funciona correctamente.

#### Experimento B1 — Función nueva, σ=2.0 (demasiado grande)
Con σ=2.0 el radio del nicho es 4 veces mayor que la distancia entre el par cercano (Δx=0.5). Todos los individuos dentro de una ventana de 2 unidades comparten nicho, lo que provoca que **los picos cercanos (x=4.5 y x=5.0) se fusionen en uno solo**. Además, dependiendo de la distribución, los picos aislados también pueden perder representación. La población se concentra principalmente en el pico de mayor altura.

#### Experimento B2 — Función nueva, σ=0.75 (valor original)
Aunque σ=0.75 funciona bien para la función original (picos separados ~3 u.), **falla con el par cercano** de la función nueva (Δx=0.5 < σ). Los picos x=4.5 y x=5.0 quedan dentro del mismo nicho y el algoritmo los trata como uno solo. Los 2 picos aislados (x=1.5 y x=8.0) sí se detectan correctamente. Se identifican **3 picos** en lugar de los 4 teóricos.

#### Experimento B3 — Función nueva, σ=0.25 (adecuado)
Con σ=0.25 < Δx=0.5, los picos cercanos están **fuera del radio de compartición** entre sí, permitiendo que el algoritmo los mantenga como nichos separados. Se identifican los **4 picos** correctamente. Este es el valor "adecuado" para esta función.

---

### ¿Cuál es el valor adecuado de SIGMA_SHARE para la función nueva?

El valor adecuado es **σ = 0.25**, porque:  
1. Es **menor que la distancia mínima entre picos** (0.5 u.), permitiendo distinguir el par cercano.  
2. Es **mayor que la resolución individual** de la población (con 100 individuos en [0,10], la resolución esperada es ~0.1 u.), por lo que no convierte a cada individuo en su propio nicho.  
3. Es suficientemente pequeño para que los 4 nichos tengan tamaños de subpoblación razonables.

**Regla general:** `SIGMA_SHARE` debería ser del orden de la **mitad de la distancia mínima entre picos** que se quiere distinguir. Para esta función: `σ_adecuado ≈ Δx_min / 2 = 0.5 / 2 = 0.25`.